# Практика 07. Обучение без учителя

**Версия:** 2026-09-25 (f95bbe8)

**Как сдавать работу**

1. Откройте ноутбук в Colab и сохраните копию себе: «Файл → Сохранить копию на Диске». Работайте в копии.
2. Выполните задания: код пишите вместо `# ВАШ КОД ЗДЕСЬ` / `raise NotImplementedError`,
   ответы на вопросы — после «*Ваш ответ:*».
3. После каждого задания запускайте ячейку с проверками. Проверки — для самоконтроля:
   их прохождение не гарантирует зачёт, а текстовые ответы проверяются отдельно.
4. Перед сдачей выполните «Среда выполнения → Перезапустить сеанс и выполнить все»: ноутбук должен
   выполниться целиком без ошибок.
5. Откройте доступ по ссылке («Настройки доступа → Все, у кого есть ссылка») и вставьте ссылку
   на свою копию в таблицу курса.

Свёрнутые ячейки со значком ▶ — служебные (загрузка данных, функции проверки). Их нужно выполнять,
но менять не нужно.

К лекции 07. План работы:

1. **Часть 1** — методы своими руками на NumPy: PCA через сингулярное разложение, метод K-средних
   (алгоритм Ллойда), силуэт. Каждый результат сверяется со scikit-learn.
2. **Часть 2** — PCA, t-SNE и K-means из scikit-learn на изображениях рукописных цифр (digits): доля объяснённой
   дисперсии и реконструкция, карты PCA и t-SNE, выбор числа кластеров и сравнение кластеров с настоящими цифрами.
3. **Часть 3** — эксперимент и выводы: K-means и DBSCAN на кластерах сложной формы, влияние радиуса DBSCAN и
   перплексии t-SNE. Код здесь простой, оценивается объяснение.

In [ ]:
# @title Служебная ячейка: импорты и функции проверки { display-mode: "form" }
import inspect
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 0
rng = np.random.default_rng(SEED)


def assert_no_loops(func):
    """Проверяет, что в теле функции нет циклов for/while (включения списков тоже считаются)."""
    source = inspect.getsource(func)
    body = source.split('"""')[-1] if '"""' in source else source
    assert not re.search(r"\b(for|while)\b", body), (
        f"В функции {func.__name__} есть цикл. Здесь нужно решение без циклов — операциями над массивами."
    )


def show_digits(images, titles=None, n_cols=10, size=0.9):
    """Рисует изображения 8 × 8 в ряд (или несколько рядов)."""
    images = np.asarray(images).reshape(-1, 8, 8)
    n_rows = int(np.ceil(len(images) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(size * n_cols, size * n_rows * 1.15), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for k, (ax, img) in enumerate(zip(axes.ravel(), images)):
        ax.imshow(img, cmap="gray_r", vmin=0, vmax=16)
        if titles is not None:
            ax.set_title(titles[k], fontsize=8)
    plt.show()


print("Готово")

# Часть 1. Методы своими руками

## Задание 1.1. PCA через сингулярное разложение

Алгоритм PCA из лекции:

1. центрировать данные: $\tilde{\mathbf{X}} = \mathbf{X} - \bar{\mathbf{x}}$ (из каждого столбца вычесть его среднее);
2. вычислить сингулярное разложение $\tilde{\mathbf{X}} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^{\top}$ (`np.linalg.svd(..., full_matrices=False)` возвращает
   `U`, вектор сингулярных чисел `s` по убыванию и матрицу `Vt` $= \mathbf{V}^{\top}$);
3. главные компоненты — первые $q$ строк `Vt` (правые сингулярные векторы);
4. дисперсия вдоль $j$-й компоненты — $\lambda_j = \sigma_j^2 / (n - 1)$.

Напишите три функции без циклов:

- `pca_fit(X, q)` — возвращает `(mean, components, explained_variance)`: вектор средних длины $d$, матрицу $q \times d$
  (компоненты по строкам, как `components_` в scikit-learn) и вектор $\lambda_1, \dots, \lambda_q$;
- `pca_transform(X, mean, components)` — координаты объектов в базисе компонент, матрица $n \times q$;
- `pca_inverse(Z, mean, components)` — восстановление объектов по координатам, матрица $n \times d$.

In [ ]:
def pca_fit(X, q):
    """(mean, components q × d, explained_variance длины q)."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError


def pca_transform(X, mean, components):
    """Координаты в базисе главных компонент."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError


def pca_inverse(Z, mean, components):
    """Восстановление объектов по координатам."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
from sklearn.decomposition import PCA

r = np.random.default_rng(1)
X_chk = r.normal(size=(200, 5)) @ r.normal(size=(5, 5)) + r.normal(size=5) * 3
mean, comps, var = pca_fit(X_chk, 3)
sk = PCA(3).fit(X_chk)
assert np.shape(mean) == (5,) and np.shape(comps) == (3, 5) and np.shape(var) == (3,), (
    f"Ожидались формы (5,), (3, 5), (3,); получено {np.shape(mean)}, {np.shape(comps)}, {np.shape(var)}"
)
assert np.allclose(mean, sk.mean_), "mean — средние по столбцам (признакам)"
assert np.allclose(comps @ comps.T, np.eye(3)), "Компоненты должны быть ортонормированы: возьмите строки Vt"
assert np.allclose(np.abs(comps @ sk.components_.T), np.eye(3), atol=1e-6), (
    "Компоненты не совпали со sklearn (с точностью до знака). Не забудьте центрировать данные перед SVD"
)
assert np.allclose(var, sk.explained_variance_), "explained_variance: sigma_j^2 / (n - 1)"
signs = np.sign(np.sum(comps * sk.components_, axis=1))
Z = pca_transform(X_chk, mean, comps)
assert np.allclose(Z * signs, sk.transform(X_chk)), "pca_transform: (X - mean) @ components.T"
assert np.allclose(pca_inverse(Z, mean, comps), sk.inverse_transform(sk.transform(X_chk))), (
    "pca_inverse: Z @ components + mean"
)
assert np.allclose(pca_inverse(pca_transform(X_chk, *pca_fit(X_chk, 5)[:2]), *pca_fit(X_chk, 5)[:2]), X_chk), (
    "При q = d восстановление должно быть точным"
)
for f in (pca_fit, pca_transform, pca_inverse):
    assert_no_loops(f)
print("OK")

## Задание 1.2. Метод K-средних

Алгоритм Ллойда чередует два шага:

- **назначение**: каждый объект относится к ближайшему центру (по евклидову расстоянию);
- **обновление**: каждый центр — среднее объектов своего кластера.

Напишите:

- `assign_clusters(X, centers)` — вектор номеров ближайших центров для всех объектов, без циклов. Квадраты
  расстояний от всех объектов до всех центров — матрица $n \times K$: удобно через broadcasting
  `X[:, None, :] - centers[None, :, :]` или через формулу $\left\lVert \mathbf{x} - \boldsymbol{\mu} \right\rVert^2 = \left\lVert \mathbf{x} \right\rVert^2 + \left\lVert \boldsymbol{\mu} \right\rVert^2 - 2\mathbf{x}^{\top}\boldsymbol{\mu}$;
- `update_centers(X, labels, K)` — матрица $K \times d$ средних по кластерам, без циклов. Подсказка: one-hot матрица
  принадлежности `np.eye(K)[labels]` размера $n \times K$; сумма объектов каждого кластера — `onehot.T @ X`. Считайте,
  что пустых кластеров нет;
- `kmeans(X, init_centers, max_iter=100)` — алгоритм Ллойда из заданных начальных центров (цикл по итерациям
  уместен). На каждой итерации: назначить объекты, пересчитать центры, записать в список `history` значение
  $\operatorname{WSS} = \sum_i \left\lVert \mathbf{x}_i - \boldsymbol{\mu}_{k(i)} \right\rVert^2$ для **новых** центров и текущих назначений. Остановиться, когда
  назначения перестали меняться (или после `max_iter` итераций). Вернуть `(centers, labels, history)`.

In [ ]:
def assign_clusters(X, centers):
    """Номер ближайшего центра для каждого объекта."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError


def update_centers(X, labels, K):
    """Средние объектов каждого кластера, матрица K × d."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
X_toy = np.array([[0.0, 0.0], [0.0, 1.0], [5.0, 5.0], [6.0, 5.0], [10.0, 0.0]])
assert np.array_equal(assign_clusters(X_toy, np.array([[0.0, 0.0], [5.0, 5.0]])), [0, 0, 1, 1, 1]), (
    "assign_clusters: каждый объект — к ближайшему центру"
)
assert np.allclose(update_centers(X_toy, np.array([0, 0, 1, 1, 1]), 2), [[0.0, 0.5], [7.0, 10 / 3]]), (
    "update_centers: центр — среднее объектов кластера"
)
assert np.allclose(update_centers(X_toy, np.array([1, 1, 0, 0, 0]), 2), [[7.0, 10 / 3], [0.0, 0.5]]), (
    "update_centers: строка k — центр кластера с номером k"
)
assert_no_loops(assign_clusters)
assert_no_loops(update_centers)
print("OK")

In [ ]:
def kmeans(X, init_centers, max_iter=100):
    """Алгоритм Ллойда. Возвращает (centers, labels, history)."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

X_bl, _ = make_blobs(400, centers=4, cluster_std=1.5, random_state=2)
init = X_bl[[0, 1, 2, 3]]
centers, labels, history = kmeans(X_bl, init)
sk = KMeans(4, init=init, n_init=1, algorithm="lloyd", tol=0).fit(X_bl)
assert np.shape(centers) == (4, 2) and np.shape(labels) == (400,), "centers — матрица 4 × 2, labels — вектор длины 400"
assert np.array_equal(labels, sk.labels_), f"Назначения расходятся со sklearn в {np.sum(labels != sk.labels_)} объектах"
assert np.allclose(centers, sk.cluster_centers_), "Центры не совпали со sklearn"
assert np.isclose(history[-1], sk.inertia_), "Последнее значение history — WSS итогового разбиения (inertia_ в sklearn)"
assert all(a >= b - 1e-9 for a, b in zip(history, history[1:])), "WSS не должна расти от итерации к итерации"
assert len(history) >= 3, "Из такой начальной расстановки алгоритму нужно несколько итераций"
_, _, h1 = kmeans(X_bl, init, max_iter=1)
assert len(h1) == 1, "При max_iter=1 должна быть ровно одна итерация"
print(f"итераций: {len(history)}, WSS: {history[0]:.0f} -> {history[-1]:.0f}")
print("OK")

## Задание 1.3. Силуэт

Для объекта $i$ из кластера $C$:

- $a(i)$ — среднее расстояние до **остальных** объектов кластера $C$ (сам объект не считается);
- $b(i)$ — наименьшее по другим кластерам $C' \neq C$ среднее расстояние до объектов $C'$;
- силуэт $s(i) = \dfrac{b(i) - a(i)}{\max\{a(i), b(i)\}}$.

Напишите `silhouette(X, labels)` — вектор $s(i)$ для всех объектов, без циклов. Метки — числа $0, \dots, K-1$,
в каждом кластере не меньше двух объектов. План:

1. матрица попарных евклидовых расстояний $\mathbf{D}$ размера $n \times n$ (не забудьте корень и `np.maximum(..., 0)`);
2. `onehot = np.eye(K)[labels]`; тогда `D @ onehot` — суммы расстояний от каждого объекта до объектов каждого
   кластера, а `onehot.sum(axis=0)` — размеры кластеров;
3. для своего кластера сумму делите на размер минус один (расстояние до себя равно нулю), для чужих — на размер;
4. чтобы найти $b(i)$, «спрячьте» свой кластер, например заменив его среднее на `np.inf`.

In [ ]:
def silhouette(X, labels):
    """Силуэт каждого объекта."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
from sklearn.metrics import silhouette_samples

s_ours = silhouette(X_bl, labels)
s_sk = silhouette_samples(X_bl, labels)
assert np.shape(s_ours) == (400,), f"Должен получиться вектор длины 400, получено {np.shape(s_ours)}"
assert np.allclose(s_ours, s_sk), (
    f"Силуэт расходится с silhouette_samples (макс. разница {np.abs(s_ours - s_sk).max():.4f}). "
    "Проверьте, что a(i) не учитывает сам объект, а b(i) берётся по ближайшему чужому кластеру"
)
X_two = np.array([[0.0], [1.0], [10.0], [12.0]])
assert np.allclose(silhouette(X_two, np.array([0, 0, 1, 1])), [1 - 1 / 11, 1 - 1 / 10, 1 - 2 / 9.5, 1 - 2 / 11.5]), (
    "Проверьте формулу на простом примере"
)
assert_no_loops(silhouette)
print(f"средний силуэт: {s_ours.mean():.3f}")
print("OK")

# Часть 2. Рукописные цифры

Набор digits из scikit-learn — 1797 изображений рукописных цифр размером 8 × 8 пикселей (UCI «Optical Recognition of
Handwritten Digits»). Каждое изображение — вектор из 64 признаков, яркость пикселя от 0 до 16. Метки цифр мы
**не используем** для обучения методов: только для того, чтобы проверить, нашли ли методы без учителя структуру,
которая с ними согласуется.

In [ ]:
# @title Загрузка данных: digits (входит в scikit-learn) { display-mode: "form" }
from sklearn.datasets import load_digits

X_digits, y_digits = load_digits(return_X_y=True)
print(f"объектов: {X_digits.shape[0]}, признаков: {X_digits.shape[1]}")
show_digits(X_digits[:20], titles=y_digits[:20])

## Задание 2.1. Доля объяснённой дисперсии

Обучите `PCA()` со всеми компонентами на `X_digits` и сохраните в `pca_full`. Посчитайте накопленную долю
объяснённой дисперсии `cum_ratio` (вектор длины 64, `np.cumsum` от `explained_variance_ratio_`) и `q95` — наименьшее число
компонент, объясняющее не меньше 95% дисперсии. Постройте график `cum_ratio` с горизонтальной линией на уровне 0.95.

Признаки не стандартизируем: все они — яркости пикселей в одной шкале. Почему это здесь правильно, вы объясните
в части 3.

In [ ]:
# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

print("q95 =", q95)

In [ ]:
assert isinstance(pca_full, PCA) and pca_full.n_components_ == 64, "pca_full — PCA со всеми 64 компонентами"
assert np.shape(cum_ratio) == (64,) and np.isclose(cum_ratio[-1], 1), "cum_ratio — накопленная сумма долей, последняя равна 1"
assert cum_ratio[q95 - 1] >= 0.95 and cum_ratio[q95 - 2] < 0.95, (
    "q95 — наименьшее число компонент с накопленной долей не меньше 0.95 (номер компоненты, а не индекс массива)"
)
print("OK")

## Задание 2.2. Реконструкция изображений

Для каждого $q$ из `Q_LIST` обучите `PCA(q)` на `X_digits`, спроецируйте изображения и восстановите их
(`transform`, затем `inverse_transform`). Сохраните в словарь `recon_mse` среднеквадратичную ошибку восстановления
по всем пикселям всех изображений: $q$ → `np.mean((X - X_rec) ** 2)`. Нарисуйте первые 10 цифр и их восстановления
для каждого $q$ (функция `show_digits`).

In [ ]:
Q_LIST = [2, 5, 10, 20, q95]
# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

print({q: round(v, 2) for q, v in recon_mse.items()})

In [ ]:
assert list(recon_mse) == Q_LIST, "В recon_mse нужны все q из Q_LIST (в том же порядке)"
values = [recon_mse[q] for q in Q_LIST]
assert all(a > b for a, b in zip(values, values[1:])), "Ошибка восстановления должна убывать с ростом q"
lam = pca_full.explained_variance_
assert np.isclose(recon_mse[10], lam[10:].sum() * (len(X_digits) - 1) / len(X_digits) / 64), (
    "recon_mse[10] не сходится с суммой отброшенных собственных значений: считайте среднее по всем пикселям всех изображений"
)
print("OK")

## Задание 2.3. Карты PCA и t-SNE

Постройте два двумерных представления цифр:

- `Z_pca` — первые две главные компоненты (`PCA(2)`);
- `Z_tsne` — t-SNE: `TSNE(n_components=2, perplexity=30, init="pca", random_state=SEED)`.

Нарисуйте обе карты рядом, раскрасив точки по настоящей цифре (`c=y_digits, cmap="tab10"`), и подпишите в
середине каждой группы её цифру (`plt.text` в медиане координат группы).

Насколько хорошо карта сохраняет структуру, можно измерить: если соседи на карте — это в основном та же цифра,
то классификатор ближайших соседей по координатам карты будет точным. Посчитайте `knn_acc` — словарь
`{"pca": ..., "tsne": ...}` со средней accuracy `KNeighborsClassifier(5)` по кросс-валидации `CV` на координатах карты.

In [ ]:
from sklearn.manifold import TSNE
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier

CV = StratifiedKFold(5, shuffle=True, random_state=SEED)
# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

print({k: round(v, 3) for k, v in knn_acc.items()})

In [ ]:
assert np.shape(Z_pca) == (1797, 2) and np.shape(Z_tsne) == (1797, 2), "Z_pca и Z_tsne — матрицы 1797 × 2"
assert np.allclose(np.abs(Z_pca), np.abs(PCA(2).fit_transform(X_digits))), "Z_pca — первые две главные компоненты X_digits"
assert set(knn_acc) == {"pca", "tsne"}, "В knn_acc нужны ключи pca и tsne"
assert np.isclose(knn_acc["pca"], cross_val_score(KNeighborsClassifier(5), Z_pca, y_digits, cv=CV).mean()), (
    "knn_acc['pca'] посчитана неверно: KNeighborsClassifier(5), cv=CV"
)
assert knn_acc["tsne"] > knn_acc["pca"] + 0.2, "На карте t-SNE соседи должны гораздо чаще совпадать по цифре, чем на PCA"
print("OK")

## Задание 2.4. Кластеризация: выбор $K$ и сравнение с цифрами

1. Для $K$ из `K_RANGE` обучите `KMeans(K, n_init=10, random_state=SEED)` на `X_digits` и сохраните списки `wss`
   (`inertia_`) и `sil` (средний силуэт, `silhouette_score`). Постройте оба графика и найдите `k_best` — $K$ с
   наибольшим силуэтом.
2. Сравните кластеры с настоящими цифрами скорректированным индексом Рэнда (`adjusted_rand_score`: 1 — полное
   совпадение разбиений, около 0 — случайное). Заполните словарь `ari` для K-means с $K = 10$
   (`KMeans(10, n_init=10, random_state=SEED)`) на трёх представлениях:
   - `"raw"` — исходные 64 признака;
   - `"pca"` — первые `q95` главных компонент;
   - `"tsne"` — координаты карты `Z_tsne`.

In [ ]:
from sklearn.metrics import adjusted_rand_score, silhouette_score

K_RANGE = list(range(2, 16))
# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

print("K с наибольшим силуэтом:", k_best)
print({k: round(v, 3) for k, v in ari.items()})

In [ ]:
assert len(wss) == len(K_RANGE) and len(sil) == len(K_RANGE), "wss и sil — по значению на каждое K из K_RANGE"
assert all(a > b for a, b in zip(wss, wss[1:])), "WSS должна убывать с ростом K"
assert np.isclose(wss[0], KMeans(2, n_init=10, random_state=SEED).fit(X_digits).inertia_), "wss — inertia_ KMeans(K, n_init=10, random_state=SEED)"
assert k_best == K_RANGE[int(np.argmax(sil))], "k_best — K с наибольшим средним силуэтом"
assert set(ari) == {"raw", "pca", "tsne"}, "В ari нужны ключи raw, pca, tsne"
assert np.isclose(ari["raw"], adjusted_rand_score(y_digits, KMeans(10, n_init=10, random_state=SEED).fit_predict(X_digits))), (
    "ari['raw'] посчитан неверно"
)
assert ari["tsne"] > ari["raw"], "На карте t-SNE кластеры K-means должны лучше совпадать с цифрами"
print("OK")

# Часть 3. Эксперимент и выводы

## Задание 3.1. K-means и DBSCAN на кластерах сложной формы

Данные — две «луны» и 30 случайных точек-выбросов вокруг них (признаки стандартизированы). Настоящие метки
лун — 0 и 1, у выбросов — $-1$.

1. Кластеризуйте данные `KMeans(2, n_init=10, random_state=SEED)` и посчитайте `ari_kmeans` — индекс Рэнда
   **только по точкам лун** (без выбросов).
2. Для каждого радиуса окрестности $h$ из `EPS_LIST` (в scikit-learn он называется `eps`) запустите
   `DBSCAN(eps=eps, min_samples=5)` и сохраните в словарь
   `dbscan_results` по ключу `eps` словарь с числом найденных кластеров `"n_clusters"` (метка $-1$ — шум, это не
   кластер), долей точек, объявленных шумом, `"noise"`, и `"ari"` — индексом Рэнда по точкам лун.
3. Нарисуйте разбиения K-means и DBSCAN при $h = 0.1, 0.2, 0.5$ (шум — серым).

In [ ]:
# @title Данные: две луны с выбросами { display-mode: "form" }
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler

X_m, y_m = make_moons(500, noise=0.07, random_state=SEED)
outliers = np.random.default_rng(SEED).uniform([-1.5, -1.0], [2.5, 1.5], size=(30, 2))
X_moons = StandardScaler().fit_transform(np.vstack([X_m, outliers]))
y_moons = np.r_[y_m, -np.ones(30, dtype=int)]
is_moon = y_moons >= 0
plt.figure(figsize=(5, 3.5))
plt.scatter(*X_moons.T, c=np.where(is_moon, y_moons, 2), cmap="brg", s=6)
plt.show()

In [ ]:
from sklearn.cluster import DBSCAN

EPS_LIST = [0.05, 0.1, 0.15, 0.2, 0.3, 0.5]
# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

print(f"K-means: ARI = {ari_kmeans:.3f}")
pd.DataFrame(dbscan_results).T.round(3)

In [ ]:
assert set(dbscan_results) == set(EPS_LIST), "В dbscan_results нужны все значения EPS_LIST"
lab = DBSCAN(eps=0.2, min_samples=5).fit_predict(X_moons)
assert dbscan_results[0.2]["n_clusters"] == len(set(lab) - {-1}), "n_clusters: число различных меток без учёта шума (-1)"
assert np.isclose(dbscan_results[0.2]["noise"], np.mean(lab == -1)), "noise — доля точек с меткой -1"
assert np.isclose(dbscan_results[0.2]["ari"], adjusted_rand_score(y_moons[is_moon], lab[is_moon])), (
    "ari — индекс Рэнда только по точкам лун (is_moon)"
)
km_ref = KMeans(2, n_init=10, random_state=SEED).fit_predict(X_moons)
assert np.isclose(ari_kmeans, adjusted_rand_score(y_moons[is_moon], km_ref[is_moon])), (
    "ari_kmeans — индекс Рэнда K-means только по точкам лун (is_moon), без выбросов"
)
assert ari_kmeans < 0.6, "K-means не должен правильно разделять луны"
assert dbscan_results[0.5]["n_clusters"] == 1, "При большом eps луны сливаются в один кластер"
assert max(v["ari"] for v in dbscan_results.values()) > 0.95, "При подходящем eps DBSCAN должен почти идеально найти луны"
print("OK")

## Задание 3.2. Перплексия t-SNE

Постройте карты t-SNE первых 600 цифр для каждой перплексии из `PERPLEXITIES`
(`TSNE(2, perplexity=p, init="pca", random_state=SEED)`) в ряд, раскрасив по цифрам. Сохраните координаты в словарь
`tsne_maps`: перплексия → матрица $600 \times 2$.

In [ ]:
PERPLEXITIES = [2, 5, 30, 100]
X_sub, y_sub = X_digits[:600], y_digits[:600]
# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

In [ ]:
assert set(tsne_maps) == set(PERPLEXITIES), "В tsne_maps нужны все перплексии"
assert all(np.shape(Z) == (600, 2) for Z in tsne_maps.values()), "Каждая карта — матрица 600 × 2"
assert not np.allclose(tsne_maps[2], tsne_maps[100]), "Карты для разных перплексий должны различаться"
print("OK")

## Задание 3.3. Выводы

Ответьте на вопросы, опираясь на свои графики и числа. Ответ на каждый вопрос — 2–4 предложения.

1. Почему для digits мы не стандартизировали признаки перед PCA? Что случилось бы со стандартизацией (подсказка:
   посмотрите на дисперсии пикселей у края изображения, `X_digits.std(axis=0)`)? А в каком случае стандартизация
   перед PCA обязательна?
2. Сколько компонент объясняют 95% дисперсии (задание 2.1)? Как выглядят восстановленные цифры при $q = 2, 5, 10$
   (задание 2.2)? Как ошибка восстановления связана с собственными значениями ковариационной матрицы?
3. Сравните карты PCA и t-SNE (задание 2.3). Какие цифры PCA смешивает? Почему t-SNE разделяет их лучше, и что
   показывает разница в `knn_acc`? Можно ли по карте t-SNE судить о том, какие цифры «ближе» друг к другу?
4. Какое $K$ выбрал бы силуэт (задание 2.4)? Совпадает ли оно с числом цифр? Почему внутренний критерий не обязан
   указывать на «настоящее» число классов? Есть ли на графике WSS отчётливый «локоть»?
5. Почему K-means на карте t-SNE совпадает с цифрами гораздо лучше, чем на исходных признаках? Почему так делать
   нужно с осторожностью?
6. Почему K-means не разделяет «луны» (задание 3.1), а DBSCAN при подходящем $h$ разделяет? Что происходит при
   слишком малом и слишком большом $h$? Как DBSCAN обошёлся с выбросами?
7. Как меняется карта t-SNE с перплексией (задание 3.2)? Что означает перплексия и почему при очень малой
   перплексии появляются мелкие «островки»?
8. По заданию 1.3: что означают силуэт около 1, около 0 и отрицательный? Где на плоскости находятся объекты с
   отрицательным силуэтом?

*Ваш ответ:*